In [7]:
import l4acados as l4a
import torch
from simple_neural_mpc.utils.misc import project_root
from neural_model_identification.learner.nn.mlp import MLP_Pinn
from neural_model_identification.parameters.train_params import TrainParams

root = project_root()
torch_model = MLP_Pinn(
    TrainParams.state_dim, TrainParams.input_dim, TrainParams.latent_dim, use_pinn=True
)
torch_model.load_state_dict(
    torch.load(
        f"{root}/src/neural_model_identification/trained_models/kin_unicycle/pinn_c_unikin.pth",
        weights_only=True,
        map_location=torch.device('cpu')
    ), 
    strict=False
)
torch_model.eval()

model = l4a.models.PyTorchResidualModel(torch_model)

In [ ]:
from acados_template import AcadosModel, AcadosOcp
from simple_neural_mpc.models.unicycle_kin.unicycle_kin_acados_neural import (
    Unicycle,
)
from simple_neural_mpc.utils.configuration.mpc_kin_config import (
    ModelPredictiveControllerConfig,
)
from simple_neural_mpc.utils.misc import load_config
from simple_neural_mpc.utils.trajectory import Circle
from simple_neural_mpc.utils.configuration.unicycle_config import UnicycleConfig

reference = Circle(freq=0.2)

# Bicycle model and corresponding controller
robot_config = load_config(
    f"{root}/config/models/unicycle.yaml",
    UnicycleConfig,
)
robot = Unicycle(config=robot_config, neural_network=torch_model)

mpc_config = load_config(
    f"{root}/config/controllers/mpc_kin.yaml",
    ModelPredictiveControllerConfig,
)

In [ ]:
l4a_solver = l4a.controllers.ResidualLearningMPC()
